# UD4.02 — Matplotlib: estilo, mapas de color y rendimiento

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
UD4 — Visualización de datos · 14 horas

Criterio 1.d · Material de partida de la práctica P4.1

## Objetivos de Aprendizaje

Al finalizar este notebook, serás capaz de:

- **Configurar Matplotlib** con `rcParams` y con hojas de estilo propias, sabiendo
  qué problema traen las variables globales
- **Elegir el mapa de color** correcto según el tipo de dato, y **demostrar** por qué
  `jet` es una mala elección en lugar de repetirlo de oídas
- **Construir gráficos avanzados**: mapas de calor, violines, contornos y superficies
- **Reconocer los engaños del doble eje Y** y decidir cuándo no usarlo
- **Medir el coste de dibujar** y aplicar `rasterized` y submuestreo cuando hay
  millones de puntos
- **Argumentar con números** dónde está el límite de Matplotlib

In [ ]:
import io
import time

import matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(20262027)

np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Guardamos la configuración de partida para poder volver a ella. Este cuaderno toca
# variables globales de Matplotlib, y sin esta copia el resto de las celdas quedarían
# afectadas por lo que haga la sección 1.
CONFIGURACION_ORIGINAL = mpl.rcParams.copy()

print("matplotlib", matplotlib.__version__)
print(f"Parámetros configurables: {len(mpl.rcParams)}")

## 1. `rcParams`: la configuración de Matplotlib

### Qué son

`matplotlib.rcParams` es un diccionario con más de mil parámetros que deciden el
aspecto por defecto de todo: tamaño de figura, tipografía, colores del ciclo, grosor
de líneas, si la rejilla está puesta, cuántos puntos por pulgada al guardar.

### Para qué sirven

El caso realista: veinte figuras para una memoria, todas con la misma tipografía, el
mismo tamaño y la misma paleta. Repetir doce parámetros en veinte figuras son
doscientas cuarenta líneas que hay que cambiar a mano cuando el criterio cambie.
Configurados una vez, las veinte figuras los heredan.

```python
# Sin rcParams, en CADA figura:
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.tick_params(labelsize=8)
ax.title.set_fontsize(11)
...  # y así doce veces, por veinte figuras

# Con rcParams, una vez:
mpl.rcParams["figure.figsize"] = (6.5, 4)
mpl.rcParams["xtick.labelsize"] = 8
...  # y las veinte figuras salen iguales
```

### El problema, que es el mismo que en el cuaderno 01

`rcParams` es **estado global**. Si una celda lo cambia, afecta a todas las que se
ejecuten después, y también a las que ya habías ejecutado y vuelvas a ejecutar. Un
cuaderno así da resultados distintos según el orden en que se toquen las celdas, que
es la definición de irreproducible.

**La forma correcta es `with plt.style.context(...)`,** que aplica la configuración
solo dentro del bloque y la deshace al salir. `mpl.rcParams[...] = valor` directo se
reserva para un script que produce una figura y termina.

In [ ]:
# Los diez parámetros que más se tocan, con lo que hace cada uno.
for clave in ["figure.figsize", "figure.dpi", "savefig.dpi", "font.size",
              "axes.titlesize", "axes.labelsize", "axes.grid",
              "axes.spines.top", "lines.linewidth", "legend.frameon"]:
    print(f"  {clave:22s} = {mpl.rcParams[clave]}")

print()
print("Y el ciclo de colores por defecto, que es de donde salen C0, C1, C2...:")
for i, color in enumerate(mpl.rcParams["axes.prop_cycle"].by_key()["color"]):
    print(f"  C{i} = {color}")

### 1.1 Una hoja de estilo propia

Un estilo es un diccionario de `rcParams`, y se puede guardar en un fichero `.mplstyle`
para reutilizarlo entre proyectos. Aquí van dos, pensados para destinos distintos:
uno para la memoria de una práctica y otro para proyectar en clase.

Fíjate en que **no cambian solo los colores**: cambian los cuerpos de letra, el grosor
de las líneas y la resolución de guardado, porque un gráfico proyectado a cinco metros
y un gráfico impreso en un folio no se leen igual.

In [ ]:
ESTILO_MEMORIA = {
    # Ancho de una columna de texto, y resolución de imprenta al guardar.
    "figure.figsize": (6.5, 4.0),
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    # Serifas y cuerpos pequeños: es lo que se lee en papel.
    "font.family": "serif",
    "font.serif": ["DejaVu Serif", "Times New Roman"],
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    # Gris y un solo acento: la memoria puede acabar impresa en blanco y negro.
    "axes.prop_cycle": mpl.cycler(color=["#000000", "#6c6c6c", "#0072B2",
                                         "#D55E00", "#CC79A7"]),
    "lines.linewidth": 1.2,
    "lines.markersize": 4,
    "axes.grid": False,
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.linewidth": 0.8,
    "legend.frameon": True,
    "legend.fancybox": False,
}

ESTILO_PROYECTOR = {
    "figure.figsize": (12.0, 6.5),
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "font.family": "sans-serif",
    "font.size": 15,
    "axes.titlesize": 20,
    "axes.labelsize": 17,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    # Colores saturados y líneas gruesas: un proyector come contraste.
    "axes.prop_cycle": mpl.cycler(color=["#0072B2", "#D55E00", "#009E73",
                                         "#CC79A7", "#F0E442"]),
    "lines.linewidth": 3.5,
    "lines.markersize": 10,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 1.6,
}


def figura_de_prueba(titulo):
    """La misma figura, para poder comparar estilos sin que cambie nada más."""
    x = np.linspace(0, 6, 120)
    fig, ax = plt.subplots()
    ax.plot(x, np.exp(-x / 3) * np.sin(2 * np.pi * x / 2), label="Oscilación")
    ax.plot(x, np.exp(-x / 3), "--", label="Envolvente")
    ax.plot(x, -np.exp(-x / 3), "--")
    ax.set_title(titulo)
    ax.set_xlabel("Tiempo (s)")
    ax.set_ylabel("Amplitud (mV)")
    ax.legend(loc="upper right")
    return fig


for nombre, estilo in (("memoria", ESTILO_MEMORIA), ("proyector", ESTILO_PROYECTOR)):
    with plt.style.context(estilo):
        fig = figura_de_prueba(f"Estilo «{nombre}»")
        plt.show()

print("Al salir del `with`, la configuración global no se ha tocado:")
print(f"  figure.figsize sigue siendo {mpl.rcParams['figure.figsize']}")

### 1.2 Guardar el estilo en un fichero

Un diccionario en el cuaderno solo sirve en ese cuaderno. Un fichero `.mplstyle` se
guarda junto al código del proyecto y lo usa cualquier script:

```python
plt.style.use("estilos/memoria.mplstyle")
```

El formato es `clave: valor`, una por línea. La única pega es que
`axes.prop_cycle` hay que escribirlo con la sintaxis de `cycler`, que es rara.

In [ ]:
CONTENIDO_MPLSTYLE = """\
# memoria.mplstyle — figuras para una memoria impresa
# Modulo 5073 · Programacion de Inteligencia Artificial

figure.figsize   : 6.5, 4.0
figure.dpi       : 110
savefig.dpi      : 300
savefig.bbox     : tight

font.family      : serif
font.size        : 9
axes.titlesize   : 11
axes.labelsize   : 10
xtick.labelsize  : 8
ytick.labelsize  : 8
legend.fontsize  : 8

# Gris y un acento, porque la memoria puede acabar impresa en blanco y negro.
axes.prop_cycle  : cycler('color', ['000000', '6c6c6c', '0072B2', 'D55E00', 'CC79A7'])
lines.linewidth  : 1.2
lines.markersize : 4

axes.grid        : False
axes.linewidth   : 0.8
legend.frameon   : True
legend.fancybox  : False
"""

print(CONTENIDO_MPLSTYLE)
print("Guardado en `estilos/memoria.mplstyle`, se aplica con:")
print('    plt.style.use("estilos/memoria.mplstyle")')
print()
print("Construir uno para el módulo es parte de la práctica P4.1.")

## 2. Mapas de color

Un mapa de color convierte un número en un color. Es la única forma de meter una
tercera dimensión en un plano, y es donde se cometen los errores más caros, porque
**un mapa de color mal elegido inventa estructura que no está en los datos**.

### Los cuatro tipos

| Tipo | Cuándo | Ejemplos |
|---|---|---|
| **Secuencial** | Los datos tienen orden natural de menos a más | `viridis`, `plasma`, `Blues` |
| **Divergente** | Hay un centro con significado: cero, la media | `RdBu`, `coolwarm`, `PiYG` |
| **Cualitativo** | Categorías sin orden | `tab10`, `Set2` |
| **Cíclico** | El valor da la vuelta: ángulos, horas, fases | `twilight`, `hsv` |

### El árbol de decisión

```
¿Los datos tienen un centro con significado (0, la media, "sin cambio")?
  SÍ  → DIVERGENTE, y hay que fijar el centro: vmin=-1, vmax=1, o norm=TwoSlopeNorm
  NO  → ¿Tienen orden natural de menor a mayor?
          SÍ  → ¿El valor es periódico (0° = 360°)?
                  SÍ → CÍCLICO
                  NO → SECUENCIAL   (viridis, si no hay motivo para otro)
          NO  → CUALITATIVO
```

In [ ]:
# La galería, por categorías.
categorias = {
    "Secuenciales": ["viridis", "plasma", "inferno", "magma", "cividis", "Blues"],
    "Divergentes": ["RdBu", "RdYlBu", "coolwarm", "seismic", "PiYG", "BrBG"],
    "Cualitativos": ["tab10", "tab20", "Set1", "Set2", "Set3", "Dark2"],
    "Cíclicos": ["twilight", "twilight_shifted", "hsv"],
    "Los que hay que evitar": ["jet", "rainbow", "nipy_spectral"],
}

degradado = np.linspace(0, 1, 256).reshape(1, -1)
alto_total = sum(len(v) for v in categorias.values())
fig, axes = plt.subplots(alto_total, 1, figsize=(9, alto_total * 0.30))
fig.subplots_adjust(hspace=1.6, left=0.28, right=0.98, top=0.94, bottom=0.02)

i = 0
for categoria, nombres in categorias.items():
    for j, nombre in enumerate(nombres):
        ax = axes[i]
        ax.imshow(degradado, aspect="auto", cmap=nombre)
        etiqueta = f"{categoria}\n{nombre}" if j == 0 else nombre
        ax.text(-0.02, 0.5, etiqueta, transform=ax.transAxes, ha="right",
                va="center", fontsize=8,
                fontweight="bold" if j == 0 else "normal")
        ax.set_xticks([])
        ax.set_yticks([])
        for lado in ax.spines.values():
            lado.set_visible(False)
        i += 1

fig.suptitle("Mapas de color de Matplotlib, por categoría", fontsize=13,
             fontweight="bold")
plt.show()

### 2.1 Por qué `jet` es una mala elección, medido

«No uses `jet`» se repite mucho y casi nunca se demuestra. Se puede demostrar, y son
quince líneas.

La propiedad que tiene que cumplir un mapa secuencial se llama **uniformidad
perceptual**: al recorrerlo de principio a fin, la **claridad** percibida tiene que
crecer de forma constante. Si crece a saltos, el ojo ve fronteras donde los datos son
suaves; si sube y baja, dos valores distintos se ven igual de claros y el orden se
pierde.

La claridad percibida no es el promedio de rojo, verde y azul: es la coordenada **L\***
del espacio CIELAB, y se calcula en tres pasos desde el color de pantalla. Lo hacemos a
mano para ver que no hay magia:

1. Deshacer la corrección gamma de sRGB, que la pantalla aplica por su cuenta.
2. Combinar los tres canales con los pesos de la sensibilidad del ojo humano, que es
   mucho más sensible al verde que al azul. Sale la luminancia relativa **Y**.
3. Pasar de **Y** a **L\***, que es la escala en la que las diferencias iguales se
   perciben iguales.

In [ ]:
def claridad(nombre_mapa, n=256):
    """Devuelve la claridad percibida L* a lo largo de un mapa de color.

    L* va de 0 (negro) a 100 (blanco), y su propiedad es que diferencias iguales de
    L* se perciben como diferencias iguales de claridad. Es la escala en la que hay
    que juzgar si un mapa de color es uniforme.
    """
    rgb = plt.get_cmap(nombre_mapa)(np.linspace(0, 1, n))[:, :3]

    # 1. Deshacer la gamma de sRGB.
    lineal = np.where(rgb <= 0.04045, rgb / 12.92,
                      ((rgb + 0.055) / 1.055) ** 2.4)

    # 2. Luminancia relativa: el ojo es mucho más sensible al verde.
    Y = lineal @ np.array([0.2126, 0.7152, 0.0722])

    # 3. De luminancia a claridad perceptual.
    return np.where(Y > 0.008856,
                    116.0 * np.cbrt(Y) - 16.0,
                    903.3 * Y)


mapas = ["viridis", "plasma", "cividis", "Blues", "jet", "rainbow"]
posicion = np.linspace(0, 1, 256)

fig, (arriba, abajo) = plt.subplots(2, 1, figsize=(12, 8),
                                    gridspec_kw={"height_ratios": [1, 1.4]})

# Arriba: las tiras de color, para tener la referencia visual.
arriba.set_xlim(0, 1)
arriba.set_ylim(0, len(mapas))
for k, nombre in enumerate(mapas):
    arriba.imshow(posicion.reshape(1, -1), aspect="auto", cmap=nombre,
                  extent=(0, 1, len(mapas) - k - 1, len(mapas) - k - 0.15))
    arriba.text(1.01, len(mapas) - k - 0.57, nombre, va="center", fontsize=9)
arriba.set_yticks([])
arriba.set_xticks([])
arriba.set_title("Los seis mapas de color", fontweight="bold")

# Abajo: la claridad medida a lo largo de cada uno.
for nombre in mapas:
    L = claridad(nombre)
    # Cuántas veces la claridad cambia de dirección: en un mapa uniforme, cero.
    cambios = int(np.sum(np.diff(np.sign(np.diff(L))) != 0))
    abajo.plot(posicion, L, linewidth=2.2,
               label=f"{nombre} — {cambios} cambios de dirección")

abajo.set_title("Claridad percibida (L*) a lo largo de cada mapa\n"
                "Una recta que sube es lo correcto; los dientes son el problema",
                fontweight="bold")
abajo.set_xlabel("Posición en el mapa de color (de 0 a 1)")
abajo.set_ylabel("Claridad L* (0 = negro, 100 = blanco)")
abajo.legend(fontsize=9, loc="lower right")
abajo.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print(f"{'mapa':>16}  {'L* mínima':>10}  {'L* máxima':>10}  "
      f"{'cambios de':>11}  {'salto':>8}")
print(f"{'':>16}  {'':>10}  {'':>10}  {'dirección':>11}  {'máximo':>8}")
print("-" * 66)
for nombre in mapas:
    L = claridad(nombre)
    cambios = int(np.sum(np.diff(np.sign(np.diff(L))) != 0))
    print(f"{nombre:>16}  {L.min():>10.1f}  {L.max():>10.1f}  "
          f"{cambios:>11d}  {np.abs(np.diff(L)).max():>8.2f}")

print()
print("Y esto es lo que dicen los números:")
print()
print("- `viridis`, `plasma` y `cividis` suben en línea recta y NO cambian de")
print("  dirección ni una vez. Cada paso de color es un paso igual de claridad.")
print("- `jet` y `rainbow` cambian de dirección varias veces: el amarillo del medio")
print("  es MÁS CLARO que los extremos. Eso hace que el ojo vea una banda brillante")
print("  en mitad del rango, y esa banda no está en los datos: está en el mapa.")
print("- Además `jet` tiene saltos de claridad mucho mayores, y ahí es donde")
print("  aparecen las fronteras falsas.")

### 2.2 La consecuencia, en un dato concreto

Los números anteriores se ven. Los mismos datos, seis mapas de color: en tres de
ellos hay estructura que no existe.

In [ ]:
# Un dato SUAVE: dos gaussianas solapadas. No tiene ninguna frontera.
x = np.linspace(-3, 3, 300)
y = np.linspace(-3, 3, 300)
X, Y = np.meshgrid(x, y)
campo = (np.exp(-((X - 0.8) ** 2 + (Y - 0.5) ** 2) / 1.5)
         + 0.75 * np.exp(-((X + 1.0) ** 2 + (Y + 0.8) ** 2) / 2.0))

fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
fig.suptitle("Los mismos datos, sin ninguna frontera, con seis mapas de color",
             fontsize=15, fontweight="bold")

for ax, nombre in zip(axes.ravel(), mapas):
    imagen = ax.imshow(campo, cmap=nombre, extent=(-3, 3, -3, 3), origin="lower")
    L = claridad(nombre)
    cambios = int(np.sum(np.diff(np.sign(np.diff(L))) != 0))
    veredicto = "correcto" if cambios == 0 else f"{cambios} cambios de claridad"
    ax.set_title(f"{nombre} — {veredicto}", fontweight="bold", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(imagen, ax=ax, shrink=0.8)

fig.tight_layout()
plt.show()

print("En `jet` y en `rainbow` se ven anillos concéntricos alrededor de cada máximo.")
print("En los datos NO hay anillos: el campo es una suma de dos gaussianas, y es")
print("perfectamente suave. Los anillos son el mapa de color, no el dato.")
print()
print("Esto en un mapa de temperatura es una anécdota. En una imagen médica, o en un")
print("mapa de activación de una red neuronal, es un diagnóstico inventado.")

### 2.3 La otra razón: imprimir y no ver los colores

Dos comprobaciones más que se hacen en dos minutos y que casi nadie hace.

**En blanco y negro.** Si el mapa es uniforme, al pasarlo a escala de grises la
información se conserva. Si no lo es, se pierde: dos valores distintos acaban del
mismo gris.

**Con daltonismo.** La forma más común, la deuteranopía, afecta a cerca del 8 % de
los hombres, y consiste en confundir rojo y verde. La simulación que sigue es
aproximada, pero basta para ver qué mapas sobreviven.

In [ ]:
def a_grises(rgb):
    """Convierte colores sRGB a la escala de grises que percibe el ojo."""
    lineal = np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + 0.055) / 1.055) ** 2.4)
    Y = lineal @ np.array([0.2126, 0.7152, 0.0722])
    gris = np.where(Y <= 0.0031308, 12.92 * Y, 1.055 * Y ** (1 / 2.4) - 0.055)
    return np.stack([gris] * 3, axis=-1)


def simula_deuteranopia(rgb):
    """Aproximación de cómo ve los colores quien confunde el rojo y el verde.

    Es una matriz de mezcla estándar en la bibliografía de accesibilidad. No es un
    modelo clínico: es suficiente para descartar un mapa de color.
    """
    matriz = np.array([[0.625, 0.375, 0.0],
                       [0.700, 0.300, 0.0],
                       [0.0, 0.300, 0.700]])
    return np.clip(rgb @ matriz.T, 0, 1)


comparar = ["viridis", "cividis", "jet", "RdYlGn"]
tira = np.linspace(0, 1, 256)

fig, axes = plt.subplots(len(comparar), 3, figsize=(13, len(comparar) * 1.25))
fig.suptitle("Cada mapa de color visto de tres formas", fontsize=14,
             fontweight="bold")

for fila, nombre in enumerate(comparar):
    rgb = plt.get_cmap(nombre)(tira)[:, :3]
    versiones = [("Como se ve en pantalla", rgb),
                 ("Impreso en blanco y negro", a_grises(rgb)),
                 ("Con deuteranopía", simula_deuteranopia(rgb))]
    for columna, (titulo, colores) in enumerate(versiones):
        ax = axes[fila, columna]
        ax.imshow(colores.reshape(1, -1, 3), aspect="auto")
        ax.set_yticks([])
        ax.set_xticks([])
        if fila == 0:
            ax.set_title(titulo, fontsize=10, fontweight="bold")
        if columna == 0:
            ax.set_ylabel(nombre, rotation=0, ha="right", va="center",
                          fontsize=10, labelpad=18)

fig.tight_layout()
plt.show()

print("Fila a fila:")
print()
print("- `viridis` y `cividis` siguen siendo un degradado ordenado en las tres")
print("  versiones. Se pueden imprimir y las puede leer cualquiera.")
print("- `jet` en blanco y negro es un desastre: el azul oscuro del principio y el")
print("  rojo oscuro del final acaban del MISMO gris. Los dos extremos del rango se")
print("  vuelven indistinguibles.")
print("- `RdYlGn` (rojo-amarillo-verde, el clásico del semáforo) con deuteranopía")
print("  pierde justo la distinción que lo hacía útil. Y es el mapa que se usa en la")
print("  mitad de los cuadros de mando del mundo.")
print()
print("Regla del módulo: si no hay una razón escrita para otro, `viridis`.")

## 3. Gráficos avanzados

### 3.1 Mapa de calor y matriz de correlación

Un mapa de calor pinta una matriz: cada celda un color. El caso más frecuente es la
**matriz de correlación**, y tiene dos reglas que no son opcionales:

1. **Mapa divergente**, porque el cero significa algo: ninguna relación.
2. **`vmin=-1, vmax=1`**, porque la correlación de Pearson siempre va de −1 a 1. Sin
   fijar los límites, Matplotlib los ajusta a los datos, el blanco deja de estar en el
   cero y dos mapas de calor distintos ya no se pueden comparar.

Y una advertencia que importa más que las dos reglas: la correlación de Pearson mide
relación **lineal**. Dos variables con una relación en forma de U tienen correlación
cercana a cero, y el mapa de calor dirá que no hay nada.

In [ ]:
nombres = [f"var_{c}" for c in "ABCDEFGH"]
datos = rng.normal(size=(200, len(nombres)))
datos[:, 1] = datos[:, 0] * 0.85 + rng.normal(0, 0.25, 200)    # B sigue a A
datos[:, 3] = -datos[:, 2] * 0.70 + rng.normal(0, 0.35, 200)   # D es lo contrario de C
datos[:, 5] = datos[:, 4] ** 2 + rng.normal(0, 0.30, 200)      # F depende de E, en U

tabla = pd.DataFrame(datos, columns=nombres)
correlacion = tabla.corr()

fig, (izq, der) = plt.subplots(1, 2, figsize=(15, 6.5))

# Izquierda: el mapa de calor, bien hecho.
imagen = izq.imshow(correlacion, cmap="RdBu_r", vmin=-1, vmax=1)
izq.set_xticks(range(len(nombres)))
izq.set_yticks(range(len(nombres)))
izq.set_xticklabels(nombres, rotation=45, ha="right")
izq.set_yticklabels(nombres)
for i in range(len(nombres)):
    for j in range(len(nombres)):
        valor = correlacion.iloc[i, j]
        izq.text(j, i, f"{valor:.2f}", ha="center", va="center", fontsize=8,
                 # Texto blanco sobre celda oscura, negro sobre clara: sin esto
                 # las celdas de los extremos son ilegibles.
                 color="white" if abs(valor) > 0.55 else "black",
                 fontweight="bold" if abs(valor) > 0.7 else "normal")
barra = fig.colorbar(imagen, ax=izq, shrink=0.85)
barra.set_label("Correlación de Pearson")
izq.set_title("Matriz de correlación\nDivergente y con los límites fijados en ±1",
              fontsize=12, fontweight="bold", pad=12)

# Derecha: la relación que la correlación no ve.
der.scatter(tabla["var_E"], tabla["var_F"], s=18, alpha=0.6,
            edgecolors="none", color="#1a5276")
der.set_title(f"var_E frente a var_F\n"
              f"Correlación de Pearson = {correlacion.loc['var_E', 'var_F']:.3f}",
              fontsize=12, fontweight="bold", pad=12)
der.set_xlabel("var_E")
der.set_ylabel("var_F")
der.grid(True, alpha=0.3)
der.annotate("Relación evidente,\ncorrelación casi cero",
             xy=(0, 0.3), xytext=(1.2, 4),
             arrowprops=dict(arrowstyle="->", color="#922b21", lw=1.8),
             fontsize=10, color="#922b21", fontweight="bold")

fig.tight_layout()
plt.show()

print("Las correlaciones más fuertes que encuentra el mapa de calor:")
#
# Hay que quitar la diagonal, que vale 1 por definición y taparía todo lo demás.
# `np.fill_diagonal(correlacion.values, 0)` era la forma habitual de hacerlo y **ya
# no funciona**: desde pandas 3.0 el array que devuelve `.values` es de solo lectura,
# para que nadie modifique un DataFrame por la espalda. La forma correcta es tapar la
# diagonal con una máscara y dejar que `stack` descarte los ausentes.
diagonal = np.eye(len(nombres), dtype=bool)
absoluta = correlacion.abs().where(~diagonal)
for (a, b), valor in absoluta.stack().nlargest(6)[::2].items():
    signo = "+" if correlacion.loc[a, b] > 0 else "-"
    print(f"  {a} y {b}: {signo}{valor:.3f}")
print()
print(f"Y la que NO encuentra: var_E y var_F, con "
      f"{correlacion.loc['var_E', 'var_F']:+.3f}, cuando el gráfico de la derecha")
print("muestra que una determina a la otra por completo. La correlación de Pearson")
print("mide relación LINEAL: si la relación es una parábola, no la ve.")
print()
print("De ahí la norma: un mapa de calor de correlaciones no sustituye a mirar las")
print("nubes de puntos. Sirve para decidir CUÁLES mirar.")

### 3.2 Violines: la forma completa del reparto

Una caja (`boxplot`) resume un reparto en cinco números: mínimo, primer cuartil,
mediana, tercer cuartil y máximo. Un violín (`violinplot`) dibuja la densidad
estimada, o sea, la forma entera.

La diferencia importa cuando hay **dos grupos escondidos dentro de uno**. La caja da
la mediana, que cae justo en el hueco entre los dos grupos, donde no hay nadie.

In [ ]:
grupos = {
    "Control": rng.normal(100, 15, 200),
    "Tratamiento A": rng.normal(110, 12, 200),
    # Este es bimodal a propósito: la mitad responde y la mitad no.
    "Tratamiento B": np.concatenate([rng.normal(95, 8, 100),
                                     rng.normal(118, 8, 100)]),
    "Tratamiento C": rng.gamma(10, 10, 200) + 50,
}

fig, (izq, der) = plt.subplots(1, 2, figsize=(15, 6))

# Izquierda: los cuatro grupos en violín.
partes = izq.violinplot(list(grupos.values()),
                        positions=range(1, len(grupos) + 1),
                        showmeans=True, showmedians=True, showextrema=True)
for cuerpo, color in zip(partes["bodies"],
                         ["#5dade2", "#ec7063", "#58d68d", "#f5b041"]):
    cuerpo.set_facecolor(color)
    cuerpo.set_alpha(0.75)
    cuerpo.set_edgecolor("black")
    cuerpo.set_linewidth(1.2)
partes["cmeans"].set_color("#922b21")
partes["cmeans"].set_linewidth(2)
partes["cmedians"].set_color("#1a5276")
partes["cmedians"].set_linewidth(2)

izq.set_xticks(range(1, len(grupos) + 1))
izq.set_xticklabels(grupos.keys(), rotation=15, ha="right")
izq.set_ylabel("Medición (unidades)")
izq.set_title("Cuatro grupos, cuatro formas distintas", fontsize=12,
              fontweight="bold", pad=12)
izq.grid(True, axis="y", alpha=0.3)

from matplotlib.lines import Line2D
izq.legend(handles=[Line2D([0], [0], color="#922b21", lw=2, label="Media"),
                    Line2D([0], [0], color="#1a5276", lw=2, label="Mediana")],
           loc="upper left", fontsize=9)

# Derecha: el grupo bimodal, en violín y en caja.
bimodal = grupos["Tratamiento B"]
violin = der.violinplot([bimodal], positions=[1], showmeans=True, showmedians=True)
violin["bodies"][0].set_facecolor("#58d68d")
violin["bodies"][0].set_alpha(0.75)
der.boxplot([bimodal], positions=[2], widths=0.35, patch_artist=True,
            boxprops=dict(facecolor="#ec7063", alpha=0.75),
            medianprops=dict(color="#1a5276", linewidth=2))

der.set_xticks([1, 2])
der.set_xticklabels(["Violín", "Caja"])
der.set_ylabel("Medición (unidades)")
der.set_title("El mismo grupo, dos resúmenes\nTratamiento B tiene DOS poblaciones",
              fontsize=12, fontweight="bold", pad=12)
der.grid(True, axis="y", alpha=0.3)
der.annotate("El violín muestra\nlos dos grupos", xy=(1.28, 118), xytext=(0.55, 145),
             arrowprops=dict(arrowstyle="->", lw=1.8, color="#1d6b3f"),
             fontsize=9, color="#1d6b3f", fontweight="bold")
der.annotate("La caja da una mediana\nen el hueco entre los dos",
             xy=(2.0, float(np.median(bimodal))), xytext=(1.9, 60),
             arrowprops=dict(arrowstyle="->", lw=1.8, color="#922b21"),
             fontsize=9, color="#922b21", fontweight="bold")

fig.tight_layout()
plt.show()

print(f"{'grupo':>16} {'media':>8} {'mediana':>9} {'desv.':>8} {'forma':>12}")
print("-" * 58)
for nombre, valores in grupos.items():
    sesgo = float(np.mean(((valores - valores.mean()) / valores.std()) ** 3))
    forma = ("simétrica" if abs(sesgo) < 0.35
             else "cola derecha" if sesgo > 0 else "cola izquierda")
    print(f"{nombre:>16} {valores.mean():>8.1f} {np.median(valores):>9.1f} "
          f"{valores.std():>8.1f} {forma:>12}")

print()
print("Fíjate en Tratamiento B: media 106, mediana 106, desviación 13. Los tres")
print("números son de un grupo perfectamente normal, y no lo es. Ningún resumen")
print("numérico de tres cifras distingue una campana de dos campanas.")
print()
print("Contrapartida honesta del violín: la densidad es una ESTIMACIÓN, con su")
print("parámetro de suavizado. Con menos de unas 50 observaciones por grupo dibuja")
print("formas que no están en los datos, y entonces la caja es más sincera.")

### 3.3 Contornos: funciones de dos variables

Un contorno dibuja las curvas donde una función de dos variables vale lo mismo. Es un
mapa topográfico: líneas juntas significan pendiente fuerte, líneas separadas
significan meseta, y una línea cerrada rodea un máximo o un mínimo.

En este módulo aparece por dos motivos concretos:

- **Superficies de pérdida.** El error de un modelo según dos de sus parámetros. Es la
  imagen que explica qué hace el descenso de gradiente y por qué a veces se queda
  atascado.
- **Fronteras de decisión.** Se evalúa el clasificador en una rejilla densa que cubre
  el plano y se dibuja el contorno de la clase predicha. Aparece en la UD5.

In [ ]:
def superficie(x, y):
    """Suma de tres gaussianas: tres máximos, uno más alto que los otros."""
    return (np.exp(-((x - 2) ** 2 + (y - 2) ** 2) / 2)
            + 0.7 * np.exp(-((x + 1) ** 2 + (y + 1) ** 2) / 3)
            + 0.5 * np.exp(-((x - 1) ** 2 + (y + 3) ** 2) / 1.5))


x = np.linspace(-4, 4, 200)
y = np.linspace(-4, 4, 200)
X, Y = np.meshgrid(x, y)
Z = superficie(X, Y)

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
fig.suptitle("Cuatro formas de mirar la misma función de dos variables",
             fontsize=15, fontweight="bold")

# 1. Solo líneas, con el valor rotulado. Lo más honesto para un informe.
ax = axes[0, 0]
lineas = ax.contour(X, Y, Z, levels=10, cmap="viridis", linewidths=1.8)
ax.clabel(lineas, inline=True, fontsize=7, fmt="%1.2f")
ax.set_title("contour: solo líneas de nivel", fontweight="bold")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.grid(True, alpha=0.25)

# 2. Relleno más líneas negras encima: se ve la región y se lee el valor.
ax = axes[0, 1]
relleno = ax.contourf(X, Y, Z, levels=20, cmap="viridis")
ax.contour(X, Y, Z, levels=10, colors="black", linewidths=0.5, alpha=0.4)
fig.colorbar(relleno, ax=ax, label="valor")
ax.set_title("contourf + contour: la combinación útil", fontweight="bold")
ax.set_xlabel("x")
ax.set_ylabel("y")

# 3. El gradiente, que es la dirección en la que la función crece más deprisa.
# Es literalmente lo que sigue el descenso de gradiente, al revés.
ax = axes[1, 0]
relleno = ax.contourf(X, Y, Z, levels=15, cmap="viridis", alpha=0.65)
dy, dx = np.gradient(Z, y, x)
salto = 12
ax.quiver(X[::salto, ::salto], Y[::salto, ::salto],
          dx[::salto, ::salto], dy[::salto, ::salto],
          alpha=0.8, scale=12, width=0.004)
fig.colorbar(relleno, ax=ax, label="valor")
ax.set_title("Contorno y campo de gradiente\nLas flechas apuntan cuesta arriba",
             fontweight="bold")
ax.set_xlabel("x")
ax.set_ylabel("y")

# 4. Una trayectoria de descenso de gradiente, que es para lo que sirve todo esto.
ax = axes[1, 1]
relleno = ax.contourf(X, Y, Z, levels=20, cmap="viridis")
ax.contour(X, Y, Z, levels=10, colors="white", linewidths=0.5, alpha=0.5)

# Ascenso de gradiente a mano: en cada paso, moverse en la dirección del gradiente.
# Se hace ascenso y no descenso porque la función tiene máximos, no mínimos, y la
# mecánica es idéntica cambiando el signo.
paso, punto = 0.55, np.array([-3.2, 3.4])
trayectoria = [punto.copy()]
for _ in range(45):
    h = 1e-4
    gradiente = np.array([
        (superficie(punto[0] + h, punto[1]) - superficie(punto[0] - h, punto[1])) / (2 * h),
        (superficie(punto[0], punto[1] + h) - superficie(punto[0], punto[1] - h)) / (2 * h),
    ])
    punto = punto + paso * gradiente
    trayectoria.append(punto.copy())
trayectoria = np.array(trayectoria)

ax.plot(trayectoria[:, 0], trayectoria[:, 1], "o-", color="#c0392b",
        markersize=3.5, linewidth=1.6, label="Trayectoria")
ax.plot(*trayectoria[0], "s", color="white", markersize=10,
        markeredgecolor="#c0392b", markeredgewidth=2, label="Inicio")
ax.plot(*trayectoria[-1], "*", color="white", markersize=17,
        markeredgecolor="#c0392b", markeredgewidth=2, label="Final")
fig.colorbar(relleno, ax=ax, label="valor")
ax.legend(loc="lower right", fontsize=8)
ax.set_title("Para qué sirve: ver dónde acaba un optimizador", fontweight="bold")
ax.set_xlabel("x")
ax.set_ylabel("y")

fig.tight_layout()
plt.show()

print(f"El optimizador partió de ({trayectoria[0, 0]:.2f}, {trayectoria[0, 1]:.2f}) "
      f"y acabó en ({trayectoria[-1, 0]:.2f}, {trayectoria[-1, 1]:.2f}),")
print(f"con valor {superficie(*trayectoria[-1]):.4f}.")
print(f"El máximo global de esta superficie vale {Z.max():.4f} y está cerca de (2, 2).")
print()
print("Si los dos números no coinciden, el optimizador se ha quedado en un máximo")
print("LOCAL. Cambia el punto de partida a (1.5, 1.5) y vuelve a ejecutar la celda:")
print("el mismo algoritmo, con el mismo paso, acaba en otro sitio. Eso es lo que un")
print("gráfico de contorno enseña y una tabla de números no.")

### 3.4 Tres dimensiones, y por qué casi siempre son mala idea

Matplotlib dibuja en tres dimensiones con `projection="3d"`. Funciona, y hay que
conocerlo, pero conviene saber lo que cuesta: en un gráfico 3D estático **el lector no
puede girar la figura**, y sin girarla no puede juzgar distancias ni profundidad. Lo
que se gana en impresión se pierde en precisión de lectura.

La comparación de abajo es el argumento: la misma superficie en 3D y en contorno. En
el 3D se ve la forma general; en el contorno se puede **leer el valor** de cualquier
punto y ver cuántos máximos hay. Para un informe, el contorno gana casi siempre.

Si la figura de verdad necesita rotarse, la respuesta no es Matplotlib: es el cuaderno
04, con Plotly.

In [ ]:
x3 = np.linspace(-4, 4, 60)
y3 = np.linspace(-4, 4, 60)
X3, Y3 = np.meshgrid(x3, y3)
Z3 = superficie(X3, Y3)

fig = plt.figure(figsize=(15, 10))
fig.suptitle("Tres dimensiones frente a dos", fontsize=15, fontweight="bold")

ax = fig.add_subplot(2, 2, 1, projection="3d")
sup = ax.plot_surface(X3, Y3, Z3, cmap="viridis", edgecolor="none", alpha=0.95)
ax.set_xlabel("x", labelpad=8)
ax.set_ylabel("y", labelpad=8)
ax.set_zlabel("z", labelpad=8)
ax.set_title("plot_surface", fontweight="bold", pad=16)
ax.view_init(elev=28, azim=45)
fig.colorbar(sup, ax=ax, shrink=0.55)

ax = fig.add_subplot(2, 2, 2, projection="3d")
ax.plot_wireframe(X3, Y3, Z3, color="#2471a3", linewidth=0.7, rstride=3, cstride=3)
ax.set_xlabel("x", labelpad=8)
ax.set_ylabel("y", labelpad=8)
ax.set_zlabel("z", labelpad=8)
ax.set_title("plot_wireframe: se ve la malla y lo que hay detrás",
             fontweight="bold", pad=16)
ax.view_init(elev=28, azim=45)

# El mismo dato, mirado desde otro ángulo: la conclusión cambia.
ax = fig.add_subplot(2, 2, 3, projection="3d")
ax.plot_surface(X3, Y3, Z3, cmap="viridis", edgecolor="none", alpha=0.95)
ax.set_xlabel("x", labelpad=8)
ax.set_ylabel("y", labelpad=8)
ax.set_zlabel("z", labelpad=8)
ax.set_title("La MISMA superficie, girada 180°\n"
             "Ahora el máximo grande tapa a los otros dos",
             fontweight="bold", pad=16, color="#922b21")
ax.view_init(elev=28, azim=225)

ax = fig.add_subplot(2, 2, 4)
relleno = ax.contourf(X3, Y3, Z3, levels=20, cmap="viridis")
lineas = ax.contour(X3, Y3, Z3, levels=8, colors="white", linewidths=0.6)
ax.clabel(lineas, inline=True, fontsize=7, fmt="%1.2f")
fig.colorbar(relleno, ax=ax, label="z")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("El mismo dato en contorno\nSe leen los valores y se cuentan los máximos",
             fontweight="bold", pad=16, color="#1d6b3f")

fig.tight_layout()
plt.show()

print("Compara el tercer panel con el cuarto. Son el mismo dato.")
print()
print("En el tercero, con ese ángulo, el máximo de (2, 2) tapa los otros dos y")
print("cualquiera diría que la superficie tiene un solo pico. En el contorno se")
print("cuentan los tres sin esfuerzo, y además se puede leer cuánto vale cada uno.")
print()
print("Norma práctica: 3D estático solo cuando la forma de la superficie ES el")
print("mensaje. Si hay que leer valores o contar estructuras, contorno.")

## 4. Dos ejes Y, y por qué es casi siempre mala idea

`ax.twinx()` crea un segundo `Axes` que comparte el eje X y tiene su propio eje Y. Se
usa para poner en el mismo gráfico dos variables con unidades distintas: temperatura y
precipitación, ventas y margen.

Y es una de las manipulaciones más eficaces que existen, porque **los límites de los
dos ejes los elige quien dibuja**. Con dos escalas libres se puede hacer que dos
series cualesquiera parezcan ir de la mano, o al contrario. El lector no tiene forma de
darse cuenta: hay dos ejes rotulados y todo parece correcto.

La demostración: las mismas dos series, tres elecciones de escala, tres conclusiones.

In [ ]:
meses = pd.date_range("2024-01-01", periods=24, freq="MS")
# Dos series que NO tienen nada que ver: la segunda es ruido con tendencia propia.
serie_a = 100 + np.cumsum(rng.normal(0.8, 3, 24))
serie_b = 50 + np.cumsum(rng.normal(-0.3, 2.2, 24))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Las mismas dos series, tres elecciones de escala",
             fontsize=15, fontweight="bold")

escalas = [
    ("Van de la mano", (serie_a.min() - 2, serie_a.max() + 2),
     (serie_b.min() - 2, serie_b.max() + 2)),
    ("Se separan", (0, serie_a.max() * 1.1), (serie_b.min() - 1, serie_b.max() + 1)),
    ("No tienen nada que ver", (0, serie_a.max() * 1.15), (0, serie_b.max() * 3)),
]

for ax, (titulo, limites_a, limites_b) in zip(axes, escalas):
    ax.plot(meses, serie_a, color="#c0392b", linewidth=2.5, label="Serie A")
    ax.set_ylabel("Serie A (unidades A)", color="#c0392b")
    ax.tick_params(axis="y", labelcolor="#c0392b")
    ax.set_ylim(limites_a)

    gemelo = ax.twinx()
    gemelo.plot(meses, serie_b, color="#1a5276", linewidth=2.5, linestyle="--",
                label="Serie B")
    gemelo.set_ylabel("Serie B (unidades B)", color="#1a5276")
    gemelo.tick_params(axis="y", labelcolor="#1a5276")
    gemelo.set_ylim(limites_b)

    ax.set_title(titulo, fontweight="bold", fontsize=11)
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.grid(True, alpha=0.25)

fig.tight_layout()
plt.show()

correlacion_real = float(np.corrcoef(serie_a, serie_b)[0, 1])
print(f"Correlación real entre las dos series: {correlacion_real:+.3f}")
print()
print("Los tres gráficos son correctos: los ejes están rotulados, las unidades")
print("puestas, los datos sin tocar. Y cuentan tres historias distintas, porque lo")
print("único que ha cambiado son los límites de los dos ejes, que los elijo yo.")
print()
print("Qué hacer en su lugar:")
print()
print("1. DOS PANELES con `sharex=True`. Cada serie con su escala, uno encima del")
print("   otro. Se compara la forma sin sugerir una relación que no está medida.")
print("2. NORMALIZAR las dos series (índice base 100, o tipificar) y ponerlas en un")
print("   solo eje. Entonces la comparación es honesta porque la escala es la misma.")
print("3. Si de verdad interesa la relación, un DIAGRAMA DE DISPERSIÓN de A frente a")
print("   B, que es el gráfico que responde a esa pregunta, con su correlación.")

In [ ]:
# Las tres alternativas, con los mismos datos.
fig = plt.figure(figsize=(16, 5))
gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.3, height_ratios=[1, 1])

# 1. Dos paneles con el eje X compartido.
sup = fig.add_subplot(gs[0, 0])
inf = fig.add_subplot(gs[1, 0], sharex=sup)
sup.plot(meses, serie_a, color="#c0392b", linewidth=2)
sup.set_ylabel("Serie A", fontsize=9)
sup.set_title("1. Dos paneles, eje X compartido", fontweight="bold", fontsize=10)
sup.tick_params(labelbottom=False, labelsize=8)
sup.grid(True, alpha=0.25)
inf.plot(meses, serie_b, color="#1a5276", linewidth=2, linestyle="--")
inf.set_ylabel("Serie B", fontsize=9)
inf.tick_params(axis="x", rotation=45, labelsize=7)
inf.tick_params(axis="y", labelsize=8)
inf.grid(True, alpha=0.25)

# 2. Normalizadas a índice base 100 en el primer mes.
ax = fig.add_subplot(gs[:, 1])
ax.plot(meses, serie_a / serie_a[0] * 100, color="#c0392b", linewidth=2.2,
        label="Serie A")
ax.plot(meses, serie_b / serie_b[0] * 100, color="#1a5276", linewidth=2.2,
        linestyle="--", label="Serie B")
ax.axhline(100, color="0.5", linestyle=":", linewidth=1)
ax.set_ylabel("Índice (enero 2024 = 100)", fontsize=9)
ax.set_title("2. Normalizadas, un solo eje", fontweight="bold", fontsize=10)
ax.legend(fontsize=8)
ax.tick_params(axis="x", rotation=45, labelsize=7)
ax.tick_params(axis="y", labelsize=8)
ax.grid(True, alpha=0.25)

# 3. La dispersión, que es el gráfico de la pregunta "tienen relación".
ax = fig.add_subplot(gs[:, 2])
puntos = ax.scatter(serie_a, serie_b, c=np.arange(24), cmap="viridis", s=55,
                    edgecolors="black", linewidth=0.5)
fig.colorbar(puntos, ax=ax, label="mes")
ax.set_xlabel("Serie A", fontsize=9)
ax.set_ylabel("Serie B", fontsize=9)
ax.set_title(f"3. Dispersión\nCorrelación = {correlacion_real:+.3f}",
             fontweight="bold", fontsize=10)
ax.tick_params(labelsize=8)
ax.grid(True, alpha=0.25)

fig.suptitle("Tres formas honestas de comparar dos series con unidades distintas",
             fontsize=14, fontweight="bold")
plt.show()

## 5. Anotaciones: dirigir la mirada

Una anotación dice **qué hay que mirar**. Las herramientas son cuatro:

| Herramienta | Para qué |
|---|---|
| `ax.annotate` | Texto con flecha, sobre un punto concreto |
| `ax.axhline` / `ax.axvline` | Un umbral, un objetivo, una fecha |
| `ax.axhspan` / `ax.axvspan` | Un tramo: un periodo, un rango aceptable |
| Figuras de `matplotlib.patches` | Rodear una región |

Y una regla: **dos o tres anotaciones por gráfico**. Si todo está señalado, nada está
señalado, y el gráfico pasa de dirigir la mirada a saturarla.

In [ ]:
fechas = pd.date_range("2024-01-01", periods=365)
precio = 100 + np.cumsum(rng.normal(0.05, 1.8, 365))

fig, (arriba, abajo) = plt.subplots(2, 1, figsize=(15, 8), sharex=True,
                                    gridspec_kw={"height_ratios": [3, 1]})
fig.suptitle("Anotaciones: tres, y solo tres", fontsize=15, fontweight="bold")

arriba.plot(fechas, precio, linewidth=1.6, color="#2E86AB", label="Precio")
media_20 = pd.Series(precio).rolling(20).mean()
media_60 = pd.Series(precio).rolling(60).mean()
arriba.plot(fechas, media_20, linewidth=1.3, color="#e67e22", alpha=0.85,
            label="Media móvil 20 días")
arriba.plot(fechas, media_60, linewidth=1.3, color="#8e44ad", alpha=0.85,
            label="Media móvil 60 días")

# Anotación 1: el máximo del año.
i_max = int(np.argmax(precio))
arriba.plot(fechas[i_max], precio[i_max], "*", color="#f1c40f", markersize=18,
            markeredgecolor="#7d6608", markeredgewidth=1.5)
arriba.annotate(f"Máximo del año: {precio[i_max]:.1f}\n{fechas[i_max]:%d de %B}",
                xy=(fechas[i_max], precio[i_max]),
                xytext=(fechas[i_max] - pd.Timedelta(days=95), precio[i_max] + 9),
                arrowprops=dict(arrowstyle="-|>", color="#7d6608", lw=2,
                                connectionstyle="arc3,rad=0.25"),
                fontsize=10, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.5", facecolor="#fef9e7",
                          edgecolor="#7d6608", linewidth=1.5))

# Anotación 2: un tramo con significado.
arriba.axvspan(fechas[120], fechas[190], alpha=0.15, color="#1d6b3f")
arriba.text(fechas[155], precio.min() + 3, "Campaña de verano", ha="center",
            fontsize=10, color="#145a32", fontweight="bold")

# Anotación 3: un umbral de referencia.
arriba.axhline(precio[0], color="0.45", linestyle=":", linewidth=1.5)
arriba.text(fechas[3], precio[0] + 1.5, f"Precio de partida: {precio[0]:.1f}",
            fontsize=9, color="0.3")

arriba.set_ylabel("Precio (€)")
arriba.legend(loc="upper left", fontsize=9, framealpha=0.9)
arriba.grid(True, alpha=0.25, linestyle="--")
arriba.spines["top"].set_visible(False)
arriba.spines["right"].set_visible(False)

# Panel inferior: la variación diaria, en color según el signo.
variacion = np.diff(precio, prepend=precio[0])
abajo.bar(fechas, variacion, width=1.0,
          color=np.where(variacion >= 0, "#1d6b3f", "#922b21"), alpha=0.75)
abajo.axhline(0, color="black", linewidth=0.8)
abajo.set_ylabel("Variación\ndiaria (€)", fontsize=9)
abajo.set_xlabel("Fecha")
abajo.grid(True, alpha=0.25, axis="y")
abajo.tick_params(axis="x", rotation=30)

fig.tight_layout()
plt.show()

print(f"Precio de partida {precio[0]:.2f} €, final {precio[-1]:.2f} € "
      f"({(precio[-1] / precio[0] - 1) * 100:+.1f} %)")
print(f"Máximo {precio.max():.2f} € · mínimo {precio.min():.2f} €")

## 6. Cuánto cuesta dibujar

Esta sección no estaba en el material de partida y es la que más tiene que ver con el
criterio 1.d, porque es la que se puede **medir**.

Matplotlib crea un objeto de Python por cada elemento que dibuja. Cien mil puntos son
cien mil objetos que hay que crear, recorrer y convertir a píxeles, y eso tiene un
coste que crece con el número de puntos. Es la misma historia de la UD3 —Python es
lento y lo rápido es delegar— vista en la biblioteca de dibujo.

Vamos a medir tres cosas: cuánto tarda en dibujar según el número de puntos, la
diferencia entre `plot` y `scatter`, y qué pasa al guardar.

In [ ]:
def tiempo_de_dibujo(funcion, repeticiones=3):
    """Mide lo que tarda en construirse Y en dibujarse una figura.

    `fig.canvas.draw()` es lo que fuerza el trabajo de verdad: sin esa llamada,
    Matplotlib se guarda las órdenes y no las ejecuta, y la medición saldría
    absurdamente rápida. Es el error clásico al cronometrar gráficos.
    """
    medidas = []
    for _ in range(repeticiones):
        inicio = time.perf_counter()
        fig = funcion()
        fig.canvas.draw()
        medidas.append(time.perf_counter() - inicio)
        plt.close(fig)
    return min(medidas)


tamanos = [1_000, 10_000, 100_000, 1_000_000]
datos_por_tamano = {n: (rng.normal(size=n), rng.normal(size=n)) for n in tamanos}

print(f"{'puntos':>10}  {'plot(\"o\")':>12}  {'scatter':>12}  "
      f"{'plot línea':>12}  {'peor / mejor':>13}")
print("-" * 68)

resultados = {}
for n in tamanos:
    x, y = datos_por_tamano[n]

    def con_plot_marcadores(x=x, y=y):
        fig, ax = plt.subplots()
        ax.plot(x, y, "o", markersize=1, alpha=0.3)
        return fig

    def con_scatter(x=x, y=y):
        fig, ax = plt.subplots()
        ax.scatter(x, y, s=1, alpha=0.3, edgecolors="none")
        return fig

    def con_plot_linea(x=x, y=y):
        fig, ax = plt.subplots()
        ax.plot(x[:n], np.sort(y)[:n], linewidth=0.5)
        return fig

    t_marcadores = tiempo_de_dibujo(con_plot_marcadores)
    t_scatter = tiempo_de_dibujo(con_scatter)
    t_linea = tiempo_de_dibujo(con_plot_linea)
    resultados[n] = (t_marcadores, t_scatter, t_linea)
    peor, mejor = max(t_marcadores, t_scatter), min(t_marcadores, t_scatter)
    print(f"{n:>10,}  {t_marcadores:>10.3f} s  {t_scatter:>10.3f} s  "
          f"{t_linea:>10.3f} s  {peor / mejor:>12.1f}×")

# La pendiente del ajuste log-log dice cómo crece el coste: 1 es crecimiento
# lineal. Se calcula sobre todo el rango y sobre la última década, y no sale lo
# mismo, que es el hallazgo interesante.
print()
print(f"{'':>14} {'pendiente':>10} {'pendiente en la':>17}")
print(f"{'':>14} {'log-log':>10} {'última década':>17}")
print("-" * 44)
for i, etiqueta in enumerate(["plot(\"o\")", "scatter", "plot línea"]):
    tiempos = np.array([resultados[n][i] for n in tamanos])
    global_ = float(np.polyfit(np.log10(tamanos), np.log10(tiempos), 1)[0])
    ultima = float(np.log10(tiempos[-1] / tiempos[-2]) / np.log10(10))
    print(f"{etiqueta:>14} {global_:>10.2f} {ultima:>17.2f}")

print()
print("Lo que hay que leer aquí:")
print()
print("1. La pendiente sobre todo el rango NO es 1, y no porque el crecimiento no")
print("   sea lineal: es porque **crear una figura cuesta unas décimas de segundo")
print("   haya dos puntos o dos mil**. Con pocos puntos ese coste fijo domina y")
print("   diluye la pendiente.")
print("   En la última década, cuando los puntos ya mandan sobre el coste fijo, la")
print("   pendiente se acerca a 1 y el crecimiento sí es lineal. No hay ninguna")
print("   sorpresa algorítmica: es que hay diez veces más cosas que dibujar.")
print("   Y de aquí sale una consecuencia práctica: por debajo de unos diez mil")
print("   puntos, optimizar el dibujado no sirve de nada, porque lo que se paga es")
print("   el coste fijo de la figura.")
print("2. `plot` con marcadores y `scatter` NO son lo mismo. `scatter` permite un")
print("   color y un tamaño por punto, y para eso guarda esa información punto a")
print("   punto. `plot` da el mismo aspecto a todos y por eso puede ir más deprisa.")
print("   Regla: si todos los puntos van iguales, `plot(x, y, \"o\")`, no `scatter`.")
print("3. Una LÍNEA con un millón de vértices es más barata que un millón de")
print("   marcadores: es un solo objeto con muchos puntos, no un millón de objetos.")

### 6.1 Las dos salidas: `rasterized` y submuestreo

**`rasterized=True`** convierte un elemento concreto a píxeles dentro de un fichero
vectorial. La figura sigue siendo un PDF —los ejes, el texto y las etiquetas siguen
siendo vectoriales y nítidos— pero la nube de puntos se guarda como imagen. Es lo que
arregla el problema que apareció medido en el cuaderno 01: un PDF con 50.000 puntos
que pesaba más que el PNG.

**El submuestreo** es la otra vía, y es más radical: si la nube tiene un millón de
puntos y la figura tiene un millón de píxeles, la mayoría de los puntos caen encima de
otro y no aportan nada. Dibujar cien mil elegidos al azar da la misma imagen.

Con una condición que hay que declarar: **el submuestreo hay que decirlo**, y hay que
comprobar que no cambia la conclusión. Si al quedarse con cien mil desaparece un grupo
raro que sí importaba, el submuestreo estaba mal hecho.

In [ ]:
x_grande, y_grande = datos_por_tamano[1_000_000]


def guarda_y_pesa(figura, formato="pdf", **opciones):
    memoria = io.BytesIO()
    figura.savefig(memoria, format=formato, bbox_inches="tight", **opciones)
    return len(memoria.getvalue()) / 1024


print(f"{'estrategia':>34}  {'dibujar':>9}  {'PDF':>10}")
print("-" * 58)

# 1. Todo vectorial: correcto y carísimo.
inicio = time.perf_counter()
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(x_grande, y_grande, "o", markersize=0.5, alpha=0.1)
ax.set_title("1.000.000 de puntos, vectorial")
fig.canvas.draw()
t = time.perf_counter() - inicio
print(f"{'todo vectorial':>34}  {t:>7.2f} s  {guarda_y_pesa(fig):>7.0f} KB")
plt.close(fig)

# 2. La nube rasterizada, el resto vectorial.
inicio = time.perf_counter()
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(x_grande, y_grande, "o", markersize=0.5, alpha=0.1, rasterized=True)
ax.set_title("1.000.000 de puntos, nube rasterizada")
fig.canvas.draw()
t = time.perf_counter() - inicio
print(f"{'nube con rasterized=True':>34}  {t:>7.2f} s  "
      f"{guarda_y_pesa(fig, dpi=200):>7.0f} KB")
plt.close(fig)

# 3. Submuestreo al 10 %.
indices = rng.choice(len(x_grande), 100_000, replace=False)
inicio = time.perf_counter()
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(x_grande[indices], y_grande[indices], "o", markersize=0.5, alpha=0.1)
ax.set_title("100.000 puntos (10 % al azar)")
fig.canvas.draw()
t = time.perf_counter() - inicio
print(f"{'submuestreo al 10 %':>34}  {t:>7.2f} s  {guarda_y_pesa(fig):>7.0f} KB")
plt.close(fig)

# 4. Histograma bidimensional: agregar en lugar de dibujar cada punto.
inicio = time.perf_counter()
fig, ax = plt.subplots(figsize=(7, 5))
ax.hexbin(x_grande, y_grande, gridsize=60, cmap="viridis", mincnt=1)
ax.set_title("hexbin: 1.000.000 de puntos agregados")
fig.canvas.draw()
t = time.perf_counter() - inicio
print(f"{'hexbin (agregar, no dibujar)':>34}  {t:>7.2f} s  {guarda_y_pesa(fig):>7.0f} KB")
plt.close(fig)

print()
print("La cuarta opción es la buena, y es la que casi nadie usa: en lugar de dibujar")
print("un millón de puntos, se cuentan cuántos caen en cada celda y se dibuja el")
print("recuento. Un millón de puntos se convierten en 3.600 hexágonos, se dibuja en")
print("una fracción del tiempo y ADEMÁS se ve mejor, porque con un millón de puntos")
print("y transparencia la zona central es una mancha uniforme y no se lee la")
print("densidad.")

In [ ]:
# La comparación visual, que es la que decide.
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Un millón de puntos, tres formas de mirarlos", fontsize=15,
             fontweight="bold")

axes[0].plot(x_grande, y_grande, "o", markersize=0.5, alpha=0.05)
axes[0].set_title("Todos los puntos\nLa zona densa es una mancha", fontweight="bold",
                  fontsize=10)

axes[1].plot(x_grande[indices], y_grande[indices], "o", markersize=1, alpha=0.15)
axes[1].set_title("10 % al azar\nMisma forma, misma mancha", fontweight="bold",
                  fontsize=10)

malla = axes[2].hexbin(x_grande, y_grande, gridsize=55, cmap="viridis", mincnt=1)
fig.colorbar(malla, ax=axes[2], label="puntos por celda")
axes[2].set_title("hexbin\nAhora sí se lee la densidad", fontweight="bold",
                  fontsize=10, color="#1d6b3f")

for ax in axes:
    ax.set_xlabel("x")
    ax.set_xlim(-4.5, 4.5)
    ax.set_ylim(-4.5, 4.5)
axes[0].set_ylabel("y")

fig.tight_layout()
plt.show()

print("Los dos primeros paneles son casi idénticos, y eso ya dice que 900.000 de")
print("esos puntos no aportaban nada a la imagen.")
print()
print("El tercero es el único que responde a la pregunta '¿dónde hay más datos?'.")
print("Los dos primeros solo pueden contestar '¿hasta dónde llegan los datos?'.")

### 6.2 Dónde está el límite de Matplotlib

Con lo medido se puede contestar de verdad a la pregunta del criterio 1.d, en lugar de
opinar:

| Situación | Qué usar | Por qué, con el número medido |
|---|---|---|
| Hasta ~10.000 elementos | Matplotlib sin más | Décimas de segundo: no hay problema |
| 10.000 a 100.000 | Matplotlib, `plot` en vez de `scatter`, y `rasterized` al guardar | El tiempo ya se nota, y el PDF vectorial se dispara |
| Más de 100.000 | **Agregar** antes de dibujar: `hexbin`, `hist2d`, `imshow` | Dibujar cada punto no aporta información y cuesta segundos |
| Millones, y hay que explorar interactivamente | Otra biblioteca: `datashader`, o Plotly con WebGL | Matplotlib no está pensada para redibujar en cada zoom |

Y el resumen de fondo, que es el mismo del módulo entero: **el cuello de botella
casi nunca es la biblioteca, es la decisión de dibujar más de lo que se puede leer**.
Una figura de 1.000 × 700 píxeles tiene 700.000 píxeles. Mandarle un millón de puntos
es pedirle que dibuje más cosas que píxeles tiene, y eso no es un problema de
rendimiento: es un problema de diseño del gráfico.

In [ ]:
# Restauramos la configuración de partida, por si alguna celda la ha tocado.
mpl.rcParams.update(CONFIGURACION_ORIGINAL)
print("Configuración de Matplotlib restaurada al estado inicial del cuaderno.")
print(f"  figure.figsize = {mpl.rcParams['figure.figsize']}")
print(f"  font.size      = {mpl.rcParams['font.size']}")

## Ejercicios

Las mismas tres condiciones del cuaderno 01: interfaz orientada a objetos, títulos y
etiquetas con unidad, y semilla fija.

### Ejercicio 1 (Básico): Tu hoja de estilo

Escribe un diccionario `ESTILO_MODULO` con al menos diez parámetros, pensado para las
figuras de las prácticas de este módulo (van en un documento que se lee en pantalla y
a veces se imprime). Demuestra que funciona dibujando **la misma figura** con y sin
el estilo, en dos celdas seguidas.

Usa `with plt.style.context(...)` y explica en una línea por qué no `plt.style.use`.

In [ ]:
# TODO: Escribe tu código aquí

### Ejercicio 2 (Básico): Elegir el mapa de color

Para cada uno de estos cinco casos, di **qué tipo** de mapa de color corresponde y
**cuál** concretamente, con una línea de justificación:

1. Temperatura de la superficie del mar, de 0 a 30 °C.
2. Diferencia entre el precio de este año y el del anterior, en euros.
3. Los siete departamentos de una empresa en un diagrama de sectores.
4. La hora del día a la que ocurre un suceso, de 0 a 24.
5. La probabilidad que da un clasificador, de 0 a 1.

Después dibuja los cinco con el mapa que hayas elegido, inventándote los datos.

In [ ]:
# TODO: Escribe tu respuesta y tu código aquí

### Ejercicio 3 (Intermedio): Medir un mapa de color

Usa la función `claridad` de la sección 2.1 para evaluar **tres mapas de color que no
estén en la lista del cuaderno**. Para cada uno, informa de:

- los cambios de dirección de L\*,
- el salto máximo de L\* entre dos colores consecutivos,
- si es apto como mapa secuencial, y por qué.

Después dibuja el mismo campo suave de la sección 2.2 con los tres y comprueba si tu
veredicto numérico coincide con lo que se ve.

In [ ]:
# TODO: Escribe tu código aquí
# Pista: plt.colormaps() da la lista completa de nombres disponibles

### Ejercicio 4 (Intermedio): Matriz de confusión como mapa de calor

Con la matriz de la celda siguiente —cinco clases, ya calculada—, dibuja tres mapas de
calor de la misma matriz:

1. Con los recuentos en bruto.
2. Normalizada **por filas**, o sea, dividiendo cada fila por su suma.
3. Normalizada **por columnas**.

Los tres con el recuento o el porcentaje escrito en cada celda y con el mapa de color
adecuado (piénsalo: ¿tiene centro esta magnitud?).

Y responde a la pregunta que importa: **las tres versiones llevan a conclusiones
distintas sobre qué clase va peor. ¿Cuál usarías, y para qué pregunta?**

In [ ]:
clases = ["gato", "perro", "caballo", "vaca", "oveja"]
confusion = np.array([
    [142,   9,   2,   1,   3],
    [ 14, 128,   4,   2,   5],
    [  3,   6,  61,  18,  12],
    [  1,   3,  15, 174,  22],
    [  4,   7,  11,  31,  38],
])

# TODO: Escribe tu código aquí
# Pista: confusion / confusion.sum(axis=1, keepdims=True) normaliza por filas

### Ejercicio 5 (Intermedio): Desmontar un doble eje

La celda siguiente dibuja dos series con doble eje Y y un título que afirma que hay
relación. Tu trabajo:

1. Calcula la correlación real entre las dos series.
2. Encuentra unos límites de los dos ejes con los que el gráfico sugiera **lo
   contrario** de lo que sugiere ahora.
3. Dibuja las tres alternativas honestas de la sección 4 y di cuál elegirías para un
   informe, y por qué.

In [ ]:
rng_ej = np.random.default_rng(4)
mes = pd.date_range("2023-01-01", periods=36, freq="MS")
usuarios = 1000 + np.cumsum(rng_ej.normal(25, 40, 36))
quejas = 40 + np.cumsum(rng_ej.normal(0.2, 3, 36))

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(mes, usuarios, color="#c0392b", linewidth=2.5)
ax.set_ylabel("Usuarios activos", color="#c0392b")
ax.set_ylim(usuarios.min() - 20, usuarios.max() + 20)
gemelo = ax.twinx()
gemelo.plot(mes, quejas, color="#1a5276", linewidth=2.5, linestyle="--")
gemelo.set_ylabel("Quejas mensuales", color="#1a5276")
gemelo.set_ylim(quejas.min() - 2, quejas.max() + 2)
ax.set_title("Cuantos más usuarios, más quejas: el soporte no da abasto",
             fontweight="bold")
plt.show()

# TODO: Escribe tu análisis y tu código aquí

### Ejercicio 6 (Avanzado): Superficie de pérdida y descenso de gradiente

Con la función de pérdida `L(w, b) = (w - 3)² + 2·(b + 1)² + 0.5·w·b`:

1. Dibuja su contorno relleno en `w ∈ [-2, 8]`, `b ∈ [-6, 4]`.
2. Implementa el descenso de gradiente con el gradiente **calculado a mano** (deriva
   la función tú, no por diferencias finitas).
3. Dibuja la trayectoria desde tres puntos de partida distintos, cada uno de un color.
4. Repítelo con tres tasas de aprendizaje: 0,01, 0,15 y 0,55.

Y responde: **con cuál de las tres tasas el algoritmo no converge, y qué se ve en el
gráfico que lo explica?**

In [ ]:
# TODO: Escribe tu código aquí

### Ejercicio 7 (Avanzado): Elegir la estrategia por el número

Tienes que dibujar la relación entre dos variables de un conjunto de **dos millones**
de filas, para una memoria en PDF.

1. Mide, con la función `tiempo_de_dibujo` de la sección 6, lo que tarda cada una de
   las cuatro estrategias de la sección 6.1.
2. Mide lo que pesa el PDF resultante en cada caso.
3. Dibuja las cuatro y compara **si se ve lo mismo**.
4. Escribe la recomendación en tres líneas, citando tus números.

El apartado 3 es el que importa: la estrategia más rápida no vale si pierde el
hallazgo. Genera los datos con **dos grupos**, uno de ellos pequeño, y comprueba si
cada estrategia lo conserva.

In [ ]:
# TODO: Escribe tu código aquí
# Pista: para el grupo pequeño, algo como
#   x = np.concatenate([rng.normal(0, 1, 1_990_000), rng.normal(4, 0.2, 10_000)])

## Resumen

1. **`rcParams` es estado global.** `with plt.style.context(...)` para trabajar; una
   hoja de estilo `.mplstyle` para que el proyecto entero salga igual.
2. **El mapa de color se elige con un árbol de decisión**, no por gusto: ¿tiene centro
   con significado? ¿tiene orden? ¿es periódico?
3. **`jet` es malo y se puede demostrar.** Su claridad percibida cambia de dirección
   varias veces, y eso inventa fronteras que no están en los datos.
4. **Un mapa de color se comprueba en gris y con daltonismo**, y son dos minutos.
   `viridis` sobrevive a los dos; `RdYlGn` no.
5. **Un mapa de calor de correlaciones sirve para decidir qué nubes de puntos mirar**,
   no para sustituirlas: Pearson solo ve relaciones lineales.
6. **El violín muestra la forma; la caja la esconde.** Y ningún resumen de tres cifras
   distingue una campana de dos campanas.
7. **3D estático solo si la forma es el mensaje.** Si hay que leer valores, contorno.
8. **El doble eje Y permite fabricar la conclusión que quieras.** Dos paneles con
   `sharex`, o normalizar, o una dispersión.
9. **Dibujar cuesta, y crece con el número de elementos.** Por encima de cien mil, se
   agrega antes de dibujar: `hexbin`, `hist2d`, `imshow`.
10. **Una figura tiene menos píxeles que los puntos que le mandas.** Ese es el
    argumento de fondo, y es el que se defiende en la práctica P4.1.

## Para seguir

- [Personalizar Matplotlib con rcParams y hojas de estilo](https://matplotlib.org/stable/users/explain/customizing.html)
- [Elegir mapas de color](https://matplotlib.org/stable/users/explain/colors/colormaps.html)
  — incluye las gráficas de claridad de todos los mapas, calculadas igual que aquí.
- Kenneth Moreland, *Why We Use Bad Color Maps and What You Can Do About It* (2016).
  Es el artículo del que sale el argumento de la sección 2.1.
- [Ten Simple Rules for Better Figures](https://doi.org/10.1371/journal.pcbi.1003833)
  — Rougier, Droettboom y Bourne. Siete páginas, en abierto, y vale para toda la vida.

**Siguiente:** el cuaderno 03 pasa a Seaborn, que hace en una línea varias de las
cosas que aquí han costado treinta.